# Opto artifact-only unit audit

This notebook audits Kilosort/Bombcell units for pulse-locked optical artifacts. By default it now derives `pulse_train_start_times` directly from `stim_df` using the first `stimulation_reachInit_stimROI_start_times` or `opto_tagging_timestamps` found, then batches observed pulse timestamps inside a 1 s window.

## Imports and setup

In [ ]:
from pathlib import Path
import sys
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams['figure.dpi'] = 120

analysis_dir = Path().resolve()
mouse_dir = analysis_dir.parent.parent
reach15_dir = analysis_dir.parent
if str(mouse_dir) not in sys.path:
    sys.path.append(str(mouse_dir))
if str(reach15_dir) not in sys.path:
    sys.path.append(str(reach15_dir))

import helper_func.prep_data as data_prep
import helper_func.nwb_data_prep as prep
import helper_func.opto_artifact_analysis as opto_art
from helper_func.post_analysis_setup import load_post_analysis_context

data_prep = importlib.reload(data_prep)
prep = importlib.reload(prep)
opto_art = importlib.reload(opto_art)

ARTIFACT_ONLY_RULE_DEFAULTS = opto_art.ARTIFACT_ONLY_RULE_DEFAULTS
apply_artifact_only_rules = opto_art.apply_artifact_only_rules
build_pulse_table = opto_art.build_pulse_table
build_pulse_table_from_stim_df = opto_art.build_pulse_table_from_stim_df
compute_opto_artifact_metrics = opto_art.compute_opto_artifact_metrics
export_phy_artifact_tsvs = opto_art.export_phy_artifact_tsvs
extract_cluster_spike_times_s = opto_art.extract_cluster_spike_times_s
load_event_start_times = opto_art.load_event_start_times
load_probe_artifact_context = opto_art.load_probe_artifact_context
merge_artifact_metrics_with_bombcell = opto_art.merge_artifact_metrics_with_bombcell

print('Notebook:', analysis_dir)
print('prep path:', Path(prep.__file__).resolve())
print('opto_art path:', Path(opto_art.__file__).resolve())


## Session and pulse-train settings

Manual train starts are still supported. If `EVENT_START_TIMES` and `EVENT_SOURCE` are both `None`, the notebook auto-builds pulse trains from `stim_df` using the simple 1 s grouping rule.

In [ ]:
ENV_PATH = None
SESSION_TO_ANALYZE = 3
TARGET_PROBE = 'A'

KS_DIR = None
SAVE_PATH = None

EVENT_SOURCE = None
EVENT_START_TIMES = None
EVENT_TIME_COLUMN = None
EVENT_STIMULUS_FILTER = None
EVENT_TIMES_ARE_SAMPLES = False

AUTO_BUILD_PULSE_TRAINS_FROM_STIM = True
AUTO_START_STIMULI = (
    'stimulation_reachInit_stimROI_start_times',
    'opto_tagging_timestamps',
)
AUTO_PULSE_STIMULUS = 'optical_timestamps'
AUTO_FALLBACK_PULSE_STIMULUS = 'opto_tagging_timestamps'
AUTO_TRAIN_LOOKAHEAD_S = 1.0
AUTO_MIN_PULSES_PER_TRAIN = 1

PULSE_COUNT = 10
PULSE_INTERVAL_MS = 10.0
POST_PULSE_WINDOW_MS = 1.0
PRE_PULSE_WINDOW_MS = 1.0

RULE_OVERRIDES = {
    # 'min_pulse_hit_rate': 0.85,
    # 'max_latency_jitter_ms': 0.15,
}

TOP_N_TO_REVIEW = 12
MAX_EVENTS_TO_PLOT = 60

WRITE_RESULTS_CSV = False
EXPORT_TO_PHY_TSV = False


## Load `.env`, resolve the session, and build `pulse_train_start_times`

In [ ]:
session_data_dic = prep.load_env(ENV_PATH)

MOUSE, BEHAVIORAL_FOLDER, NP_FILE, NWB_FILE, DATE, SESSION, BOMBCELL = prep.session_to_analyze(
    session_data_dic['MOUSE'], session_data_dic['BEHAVIORAL_FOLDER'],
    session_data_dic['NP_FILE'], session_data_dic['NWB_FILE'], session_data_dic['DATE'], session_data_dic['SESSION'], session_data_dic['BOMBCELL'],
    session_data_dic['NP_FILE_01'], session_data_dic['NWB_FILE_01'], session_data_dic['DATE_01'], session_data_dic['SESSION_01'], session_data_dic['BOMBCELL_01'],
    session_data_dic['NP_FILE_02'], session_data_dic['NWB_FILE_02'], session_data_dic['DATE_02'], session_data_dic['SESSION_02'], session_data_dic['BOMBCELL_02'],
    session_selection=SESSION_TO_ANALYZE,
)

BOMBCELL_ROOT_FOR_AUTO_BUILD, NWB_PATH, CONFIG_FILE, SESSION_NAME = data_prep.set_bc_paths(
    session_data_dic,
    MOUSE,
    NWB_FILE,
    NP_FILE,
    BOMBCELL,
    BEHAVIORAL_FOLDER,
    DATE,
    SESSION,
    SESSION_TO_ANALYZE,
)

staging_root = Path('H:/Grant/Neuropixels/Kilosort_Recordings') / NP_FILE / 'bombcell' / BOMBCELL
root_recording_folder = Path('H:/Grant/Neuropixels/Kilosort_Recordings') / NP_FILE
recording_folder = root_recording_folder / 'Record Node 103' / 'experiment1' / 'recording1' / 'continuous'
behavioral_folder = Path('G:/Grant/behavior_data/DLC_net') / BEHAVIORAL_FOLDER / DATE / SESSION

ctx = load_post_analysis_context(CONFIG_FILE)
probe_letters = list(ctx['probeLetters'])

if KS_DIR is None:
    ks_dir = staging_root / f'kilosort4_{TARGET_PROBE}'
else:
    ks_dir = Path(KS_DIR)

save_path = Path(SAVE_PATH) if SAVE_PATH is not None else ks_dir / 'bombcell'
probe_ctx = load_probe_artifact_context(ks_dir, save_path=save_path)
sample_rate_hz = probe_ctx['sample_rate_hz']

stim_df = None
pulse_build_mode = None
pulse_table_source = None
auto_pulse_info = None

if EVENT_START_TIMES is not None:
    event_start_times_s = load_event_start_times(
        EVENT_START_TIMES,
        times_are_samples=EVENT_TIMES_ARE_SAMPLES,
        sample_rate_hz=sample_rate_hz,
    )
    pulse_train_start_times = event_start_times_s.copy()
    pulse_table = build_pulse_table(
        pulse_train_start_times,
        pulse_count=PULSE_COUNT,
        inter_pulse_interval_s=PULSE_INTERVAL_MS / 1000.0,
    )
    pulse_build_mode = 'manual_event_start_times'
    pulse_table_source = 'synthetic_fixed_interval'
elif EVENT_SOURCE is not None:
    event_start_times_s = load_event_start_times(
        EVENT_SOURCE,
        time_col=EVENT_TIME_COLUMN,
        stimulus_filter=EVENT_STIMULUS_FILTER,
        times_are_samples=EVENT_TIMES_ARE_SAMPLES,
        sample_rate_hz=sample_rate_hz,
    )
    pulse_train_start_times = event_start_times_s.copy()
    pulse_table = build_pulse_table(
        pulse_train_start_times,
        pulse_count=PULSE_COUNT,
        inter_pulse_interval_s=PULSE_INTERVAL_MS / 1000.0,
    )
    pulse_build_mode = 'event_source_fixed_interval'
    pulse_table_source = str(EVENT_SOURCE)
elif AUTO_BUILD_PULSE_TRAINS_FROM_STIM:
    processed_bundle_dir = analysis_dir / 'processed_data' / NP_FILE
    processed_bundle_dir.mkdir(parents=True, exist_ok=True)

    bundle, merged_dic, stim_df, df_stim, pca_event_meta_bundle, extras, meta, PROCESSED_BUNDLE_DIR = prep.load_or_build_processed_bundle(
        processed_bundle_dir=processed_bundle_dir,
        nwb_path_for_auto_build=NWB_PATH,
        bombcell_root_for_auto_build=BOMBCELL_ROOT_FOR_AUTO_BUILD,
        use_bombcell_if_available=True,
        auto_build_bundle_if_missing=True,
        auto_rebuild_if_bombcell_missing=True,
        required_filenames=('merged_dic.pkl', 'stim_df.pkl', 'pca_event_meta.pkl'),
        verbose=True,
    )

    auto_pulse_info = build_pulse_table_from_stim_df(
        stim_df,
        start_stimuli=AUTO_START_STIMULI,
        pulse_stimulus=AUTO_PULSE_STIMULUS,
        fallback_pulse_stimulus=AUTO_FALLBACK_PULSE_STIMULUS,
        look_ahead_s=AUTO_TRAIN_LOOKAHEAD_S,
        min_pulses_per_train=AUTO_MIN_PULSES_PER_TRAIN,
        time_col=EVENT_TIME_COLUMN,
    )
    pulse_table = auto_pulse_info['pulse_table'].copy()
    pulse_train_start_times = auto_pulse_info['pulse_train_start_times'].copy()
    event_start_times_s = pulse_train_start_times.copy()
    pulse_build_mode = 'auto_from_stim_df'
    pulse_table_source = auto_pulse_info['pulse_source_stimulus']
else:
    raise ValueError('Set EVENT_START_TIMES, EVENT_SOURCE, or leave AUTO_BUILD_PULSE_TRAINS_FROM_STIM=True.')

pulse_train_summary = pulse_table.groupby('trial_index').size().rename('n_pulses').reset_index()

print('CONFIG_FILE:', CONFIG_FILE)
print('SESSION_NAME:', SESSION_NAME)
print('NWB_PATH:', NWB_PATH)
print('BOMBCELL_ROOT_FOR_AUTO_BUILD:', BOMBCELL_ROOT_FOR_AUTO_BUILD)
print('staging_root:', staging_root)
print('root_recording_folder:', root_recording_folder)
print('recording_folder:', recording_folder)
print('behavioral_folder:', behavioral_folder)
print('available probes:', probe_letters)
print('ks_dir:', ks_dir)
print('save_path:', save_path)
print('sample_rate_hz:', sample_rate_hz)
print('pulse_build_mode:', pulse_build_mode)
print('pulse_table_source:', pulse_table_source)
print('n_pulse_trains:', pulse_train_start_times.size)
print('n_pulses_total:', len(pulse_table))
print('recording_duration_s:', round(probe_ctx['recording_duration_s'], 3))

if auto_pulse_info is not None:
    relevant_stimuli = list(dict.fromkeys([*AUTO_START_STIMULI, AUTO_PULSE_STIMULUS, AUTO_FALLBACK_PULSE_STIMULUS]))
    display(
        stim_df.loc[stim_df['stimulus'].astype(str).isin(relevant_stimuli), 'stimulus']
        .value_counts()
        .rename_axis('stimulus')
        .to_frame('n_timestamps')
    )
    print('n_candidate_start_times:', auto_pulse_info['candidate_start_times_s'].size)
    print('n_collapsed_train_seed_times:', auto_pulse_info['train_seed_times_s'].size)

display(pd.DataFrame({'pulse_train_start_times_s': pulse_train_start_times}).head(20))
display(pulse_train_summary.head(20))
display(pulse_train_summary['n_pulses'].describe())
display(pulse_table.head(30))


## Compute per-unit timing-lock metrics

In [ ]:
artifact_metrics = compute_opto_artifact_metrics(
    spike_times_samples=probe_ctx['spike_times_samples'],
    spike_clusters=probe_ctx['spike_clusters'],
    cluster_ids=probe_ctx['cluster_ids'],
    sample_rate_hz=sample_rate_hz,
    pulse_table=pulse_table,
    post_pulse_window_s=POST_PULSE_WINDOW_MS / 1000.0,
    pre_pulse_window_s=PRE_PULSE_WINDOW_MS / 1000.0,
    recording_duration_s=probe_ctx['recording_duration_s'],
)

applied_rules = dict(ARTIFACT_ONLY_RULE_DEFAULTS)
applied_rules.update(RULE_OVERRIDES)

artifact_metrics = apply_artifact_only_rules(artifact_metrics, **RULE_OVERRIDES)
results_df = merge_artifact_metrics_with_bombcell(
    artifact_metrics,
    probe_ctx['quality_metrics_df'],
)

display(pd.Series(applied_rules, name='value').rename_axis('rule'))
display(results_df.head(10))


## Summary tables and global views

In [ ]:
candidate_mask = results_df['artifact_only_candidate'].fillna(False)
candidate_df = results_df.loc[candidate_mask].copy()
good_candidate_df = candidate_df.loc[candidate_df['bc_unitType'].astype(str).eq('GOOD')].copy()

summary_df = pd.DataFrame(
    {
        'n_units_total': [len(results_df)],
        'n_artifact_only_candidates': [len(candidate_df)],
        'n_good_and_artifact_only': [len(good_candidate_df)],
        'candidate_fraction': [len(candidate_df) / len(results_df) if len(results_df) else np.nan],
    }
)
display(summary_df)

display(
    results_df.groupby(['artifact_only_candidate', 'bc_unitType'], dropna=False)
    .size()
    .reset_index(name='n_units')
    .sort_values(['artifact_only_candidate', 'n_units'], ascending=[False, False])
)

review_cols = [
    'cluster_id',
    'artifact_only_candidate',
    'artifact_score',
    'bc_unitType',
    'bc_classificationReason',
    'pulse_hit_rate',
    'fraction_spikes_locked',
    'mean_trial_pulse_fraction',
    'median_latency_ms',
    'latency_jitter_ms',
    'off_pulse_fraction',
    'pre_to_post_ratio',
    'nSpikes_total',
    'nPeaks',
    'nTroughs',
    'spatialDecaySlope',
    'waveformBaselineFlatness',
    'scndPeakToTroughRatio',
    'artifact_only_reason',
]
display(candidate_df[review_cols].head(TOP_N_TO_REVIEW))
if not good_candidate_df.empty:
    print('Bombcell GOOD + artifact-only candidates:')
    display(good_candidate_df[review_cols].head(TOP_N_TO_REVIEW))

plot_df = results_df.copy()
colors = np.where(plot_df['artifact_only_candidate'].fillna(False), 'tab:red', '0.7')
sizes = 20 + 60 * plot_df['artifact_score'].fillna(0.0).to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].scatter(
    plot_df['pulse_hit_rate'],
    plot_df['fraction_spikes_locked'],
    c=colors,
    s=sizes,
    alpha=0.8,
    linewidths=0,
)
axes[0].set_xlabel('Pulse hit rate')
axes[0].set_ylabel('Fraction of spikes inside post-pulse window')
axes[0].set_title('Pulse locking vs spike locking')

axes[1].scatter(
    plot_df['median_latency_ms'],
    plot_df['latency_jitter_ms'],
    c=colors,
    s=sizes,
    alpha=0.8,
    linewidths=0,
)
axes[1].set_xlabel('Median latency after pulse (ms)')
axes[1].set_ylabel('Latency jitter (ms)')
axes[1].set_title('Immediate and low-jitter units rise to the top')

top_annotate = candidate_df.head(min(10, len(candidate_df)))
for _, row in top_annotate.iterrows():
    axes[0].annotate(int(row['cluster_id']), (row['pulse_hit_rate'], row['fraction_spikes_locked']), fontsize=8)
    axes[1].annotate(int(row['cluster_id']), (row['median_latency_ms'], row['latency_jitter_ms']), fontsize=8)

plt.show()


## Review plots for top candidate units

In [ ]:
def plot_event_train_raster(cluster_id, *, pre_s=0.002, post_s=0.005, max_events=60, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))

    spike_times_s = extract_cluster_spike_times_s(
        probe_ctx['spike_times_samples'],
        probe_ctx['spike_clusters'],
        cluster_id,
        sample_rate_hz,
    )
    event_df = pulse_table[['trial_index', 'event_start_time_s']].drop_duplicates().reset_index(drop=True)
    pulse_offsets_ms = pulse_table[['pulse_index', 'pulse_offset_s']].drop_duplicates()['pulse_offset_s'].to_numpy() * 1000.0
    train_end_s = float(pulse_table['pulse_offset_s'].max()) + post_s if not pulse_table.empty else post_s

    for row_idx, event_start_s in enumerate(event_df['event_start_time_s'].head(max_events).to_numpy()):
        rel_s = spike_times_s[(spike_times_s >= event_start_s - pre_s) & (spike_times_s <= event_start_s + train_end_s)] - event_start_s
        if rel_s.size:
            ax.vlines(rel_s * 1000.0, row_idx + 0.6, row_idx + 1.4, color='k', linewidth=0.6)

    for pulse_offset_ms in pulse_offsets_ms:
        ax.axvline(pulse_offset_ms, color='tab:red', linestyle='--', linewidth=0.8, alpha=0.6)

    ax.set_xlabel('Time from train start (ms)')
    ax.set_ylabel('Trial index')
    ax.set_title(f'Cluster {cluster_id}: event-aligned pulse-train raster')
    ax.set_xlim(-pre_s * 1000.0, train_end_s * 1000.0)


def plot_collapsed_pulse_raster(cluster_id, *, pre_s=0.001, post_s=0.003, max_events=60, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))

    spike_times_s = extract_cluster_spike_times_s(
        probe_ctx['spike_times_samples'],
        probe_ctx['spike_clusters'],
        cluster_id,
        sample_rate_hz,
    )
    trial_ids_to_plot = pulse_table['trial_index'].drop_duplicates().head(max_events).to_numpy()
    pulses_to_plot = pulse_table[pulse_table['trial_index'].isin(trial_ids_to_plot)].reset_index(drop=True)
    for pulse_row, pulse_time_s in enumerate(pulses_to_plot['pulse_time_s'].to_numpy()):
        rel_s = spike_times_s[(spike_times_s >= pulse_time_s - pre_s) & (spike_times_s <= pulse_time_s + post_s)] - pulse_time_s
        if rel_s.size:
            ax.vlines(rel_s * 1000.0, pulse_row + 0.6, pulse_row + 1.4, color='k', linewidth=0.5)

    ax.axvline(0.0, color='tab:red', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Time from pulse (ms)')
    ax.set_ylabel('Pulse row')
    ax.set_title(f'Cluster {cluster_id}: collapsed pulse-centered raster')
    ax.set_xlim(-pre_s * 1000.0, post_s * 1000.0)


review_df = candidate_df.copy()
if review_df.empty:
    review_df = results_df.copy()
    print('No units passed the current artifact-only rule set. Reviewing the top-ranked units instead.')

for cluster_id in review_df['cluster_id'].head(TOP_N_TO_REVIEW).astype(int):
    row = results_df.loc[results_df['cluster_id'].astype(int) == int(cluster_id)].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True)
    plot_event_train_raster(cluster_id, max_events=MAX_EVENTS_TO_PLOT, ax=axes[0])
    plot_collapsed_pulse_raster(cluster_id, max_events=MAX_EVENTS_TO_PLOT, ax=axes[1])
    fig.suptitle(
        (
            f"cluster {cluster_id} | Bombcell={row.get('bc_unitType', 'NA')} | "
            f"artifact={bool(row['artifact_only_candidate'])} | score={row['artifact_score']:.3f} | "
            f"hit_rate={row['pulse_hit_rate']:.3f} | locked={row['fraction_spikes_locked']:.3f} | "
            f"latency={row['median_latency_ms']:.3f} ms | jitter={row['latency_jitter_ms']:.3f} ms | "
            f"off_pulse_fraction={row['off_pulse_fraction']:.3f}"
        ),
        fontsize=10,
    )
    plt.show()


## Optional export

In [ ]:
if WRITE_RESULTS_CSV:
    csv_path = save_path / 'opto_artifact_audit.csv'
    results_df.to_csv(csv_path, index=False)
    print('Wrote:', csv_path)

if EXPORT_TO_PHY_TSV:
    export_paths = export_phy_artifact_tsvs(ks_dir, results_df)
    for label, export_path in export_paths.items():
        print(f'{label}: {export_path}')
else:
    print('EXPORT_TO_PHY_TSV is False. No cluster_artifactOnly*.tsv files were written.')
